In [4]:
from nilearn.decoding import SearchLight
from nilearn.input_data import NiftiMasker
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut
from nilearn.image import concat_imgs, load_img
import numpy as np
import pandas as pd
import os

# -------- Parameters --------
radius = 3  # searchlight radius in voxels
mask_img = load_img('/Users/liuq13/SPCD_data/masks/gm_0.1.nii')

In [5]:
# -------- Load all beta images --------
def load_betas(sub, run):
    tsv_file = os.path.join(tsv_dir, f'{sub}_task-odor_run-{run}_events.tsv')
    events = pd.read_csv(tsv_file, sep='\t')
    events = events[events['trial_type'].isin(['odor_1', 'odor_2'])].copy()
    events['label'] = events['trial_type'].map({'odor_1': 0, 'odor_2': 1})

    imgs = []
    for i in range(len(events)):
        trial_num = i + 1
        beta_file = os.path.join(beta_dir, f'beta_run{run}_trial{trial_num:02d}.nii')
        imgs.append(load_img(beta_file))
    return imgs, events['label'].values

    

In [6]:
# Load run 1 and 2
sub = "sub-01"  # or set dynamically

base_dir = '/Users/liuq13/SPCD_data'
beta_dir = os.path.join(base_dir, 'GLM', 'odor_scan_trialwise_2mm', sub)
tsv_dir = os.path.join(base_dir, 'nii', sub, 'func_PAID', 'odor_scan')

imgs1, y1 = load_betas(sub, run=1)
imgs2, y2 = load_betas(sub, run=2)

# Stack all beta maps and labels
all_imgs = concat_imgs(imgs1 + imgs2)
all_y = np.concatenate([y1, y2])
groups = np.array([1]*len(y1) + [2]*len(y2))  # group labels for cross-run

In [7]:
# -------- Run searchlight --------
searchlight = SearchLight(
    mask_img=mask_img,
    radius=radius,
    estimator=SVC(kernel='linear'),
    n_jobs=-1,  # use all CPUs
    cv=LeaveOneGroupOut(),
    verbose=1,
    scoring='accuracy'
)

searchlight.fit(all_imgs, all_y, groups)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 12 concurrent workers.


[LokyProcess.run] Job #5, processed 0/16507 steps (0.00%, 62.7 seconds remaining)

[LokyProcess.run] Job #5, processed 10/16507 steps (0.06%, 409.4 seconds remaining)

[LokyProcess.run] Job #5, processed 20/16507 steps (0.12%, 228.1 seconds remaining)

[LokyProcess.run] Job #5, processed 30/16507 steps (0.18%, 167.4 seconds remaining)

[LokyProcess.run] Job #5, processed 40/16507 steps (0.24%, 138.1 seconds remaining)

[LokyProcess.run] Job #5, processed 50/16507 steps (0.30%, 117.9 seconds remaining)

[LokyProcess.run] Job #5, processed 60/16507 steps (0.36%, 106.6 seconds remaining)

[LokyProcess.run] Job #5, processed 70/16507 steps (0.42%, 98.0 seconds remaining)

[LokyProcess.run] Job #5, processed 80/16507 steps (0.48%, 101.2 seconds remaining)

[LokyProcess.run] Job #5, processed 90/16507 steps (0.55%, 94.7 seconds remaining)

[LokyProcess.run] Job #5, processed 100/16507 steps (0.61%, 89.4 seconds remaining)

[LokyProcess.run] Job #5, processed 110/16507 steps (0.67%, 86.4 seco

[Parallel(n_jobs=-1)]: Done   2 out of  12 | elapsed:   52.5s remaining:  4.4min


[LokyProcess.run] Job #2, processed 13150/16507 steps (79.66%, 10.4 seconds remaining)

[LokyProcess.run] Job #2, processed 13160/16507 steps (79.72%, 10.4 seconds remaining)

[LokyProcess.run] Job #2, processed 13170/16507 steps (79.78%, 10.4 seconds remaining)

[LokyProcess.run] Job #2, processed 13180/16507 steps (79.84%, 10.3 seconds remaining)

[LokyProcess.run] Job #2, processed 13190/16507 steps (79.91%, 10.3 seconds remaining)

[LokyProcess.run] Job #2, processed 13200/16507 steps (79.97%, 10.3 seconds remaining)

[LokyProcess.run] Job #2, processed 13210/16507 steps (80.03%, 10.2 seconds remaining)

[LokyProcess.run] Job #2, processed 13220/16507 steps (80.09%, 10.2 seconds remaining)

[LokyProcess.run] Job #2, processed 13230/16507 steps (80.15%, 10.2 seconds remaining)

[LokyProcess.run] Job #2, processed 13240/16507 steps (80.21%, 10.1 seconds remaining)

[LokyProcess.run] Job #2, processed 13250/16507 steps (80.27%, 10.1 seconds remaining)

[LokyProcess.run] Job #2, proces

[Parallel(n_jobs=-1)]: Done  12 out of  12 | elapsed:   55.5s finished


SearchLight(cv=LeaveOneGroupOut(), estimator=SVC(kernel='linear'),
            mask_img=<nibabel.nifti1.Nifti1Image object at 0x3188a4df0>,
            n_jobs=-1, radius=3, scoring='accuracy', verbose=1)

In [8]:
# -------- Save result --------
from nilearn.image import new_img_like
searchlight_img = new_img_like(mask_img, searchlight.scores_)
searchlight_img.to_filename(f'/Users/liuq13/SPCD_data/GLM/searchlight_{sub}_odor_decoding.nii.gz')

[LokyProcess.run] Job #12, processed 14850/16506 steps (89.97%, 4.9 seconds remaining)

[LokyProcess.run] Job #12, processed 14860/16506 steps (90.03%, 4.9 seconds remaining)

[LokyProcess.run] Job #12, processed 14870/16506 steps (90.09%, 4.9 seconds remaining)

[LokyProcess.run] Job #12, processed 14880/16506 steps (90.15%, 4.8 seconds remaining)

[LokyProcess.run] Job #12, processed 14890/16506 steps (90.21%, 4.8 seconds remaining)

[LokyProcess.run] Job #12, processed 14900/16506 steps (90.27%, 4.8 seconds remaining)

[LokyProcess.run] Job #12, processed 14910/16506 steps (90.33%, 4.8 seconds remaining)

[LokyProcess.run] Job #12, processed 14920/16506 steps (90.39%, 4.7 seconds remaining)

[LokyProcess.run] Job #12, processed 14930/16506 steps (90.45%, 4.7 seconds remaining)

[LokyProcess.run] Job #12, processed 14940/16506 steps (90.51%, 4.7 seconds remaining)

[LokyProcess.run] Job #12, processed 14950/16506 steps (90.57%, 4.6 seconds remaining)

[LokyProcess.run] Job #12, proce